# Guía práctica de modelos de Machine Learning

Una guía para elegir un primer modelo, preparar los datos correctamente y entender qué significa el resultado. No existe un modelo que gane siempre: hay que probar opciones razonables con una evaluación justa.

## 1. Antes de elegir un modelo

### ¿Qué quieres averiguar?

- **Regresión:** quieres predecir una cantidad numérica, como precio, ventas o temperatura.
- **Clasificación:** quieres asignar una etiqueta conocida, como fraude/no fraude o especie A/B/C.
- **Clustering:** no tienes etiquetas; quieres descubrir grupos de observaciones parecidas.
- **Reducción de dimensión:** tienes muchas variables y quieres resumirlas o visualizarlas con menos dimensiones.

En los dos primeros casos hay una respuesta objetivo conocida (`y`): es **aprendizaje supervisado**. Clustering y PCA no usan una respuesta objetivo: son **no supervisados**.

### Conoce tus datos antes de entrenar

1. Comprueba cuántas filas y columnas tienes, sus tipos, valores ausentes, duplicados y rangos.
2. Mira cómo se distribuye la variable objetivo. En clasificación, comprueba si las clases están equilibradas.
3. Pregúntate si cada variable estará disponible en el momento real de predecir. Una variable que solo aparece después del resultado puede filtrar la respuesta (*data leakage*).
4. Separa los datos en entrenamiento y prueba. Entrena y toma decisiones con entrenamiento; reserva prueba para la comprobación final.
5. Si hay fechas, personas, clientes o centros repetidos, divide respetando el tiempo o los grupos. Una partición aleatoria podría colocar casos casi idénticos a ambos lados y dar una puntuación demasiado optimista.

### Conceptos que aplican a todos los modelos

- **Baseline:** resultado sencillo de referencia. En regresión puede ser predecir la media; en clasificación, la clase mayoritaria. Un modelo complejo debe mejorar ese punto de partida.
- **Entrenamiento:** el modelo ajusta sus parámetros a los ejemplos que recibe.
- **Validación:** estima cómo funcionaría con datos no vistos y sirve para comparar modelos y ajustar hiperparámetros.
- **Prueba (test):** evaluación final, una vez tomada la decisión. Si la consultas muchas veces para elegir, deja de ser una prueba imparcial.
- **Sobreajuste:** el modelo aprende detalles y ruido del entrenamiento, pero falla en datos nuevos. Suele verse como buen resultado en train y peor resultado en validación.
- **Subajuste:** el modelo es demasiado simple o está mal preparado; falla tanto en train como en validación.
- **Hiperparámetro:** ajuste elegido por quien entrena (por ejemplo, profundidad máxima de un árbol). No se aprende directamente como un coeficiente a partir de cada ejemplo.
- **Validación cruzada:** divide entrenamiento en varios pliegues, entrena y valida repetidamente. Da una comparación más estable cuando hay pocos datos. Para datos temporales se usa una estrategia que respeta el orden.

## 2. Prepara los datos sin contaminar la evaluación

- **Valores ausentes:** imputa con una regla coherente (mediana para numéricas, moda para categóricas, por ejemplo) o elimina filas/columnas si está justificado. Aprende la regla usando solo entrenamiento.
- **Variables categóricas:** One-Hot crea una columna binaria por categoría (k columnas, o k−1 si se elimina una para evitar colinealidad) y funciona bien con modelos lineales y muchos modelos de sklearn. Con categorías de alta cardinalidad genera muchas columnas; en modelos de árboles no siempre es la mejor opción. La codificación ordinal solo tiene sentido cuando las categorías tienen un orden real.
- **Escalado:** centra y/o cambia la escala de variables numéricas. Importa para modelos basados en distancia o magnitud (KNN, SVM, regresión regularizada, MLP, PCA y K-Means). Los árboles suelen ser poco sensibles a la escala.
- **Pipeline:** encadena imputación, codificación, escalado y modelo. Al validarlo, cada transformación se aprende solo con la parte de entrenamiento de cada pliegue y se evita fuga de información.
- **Desbalance de clases:** si el 98 % de ejemplos son negativos, predecir siempre negativo ya da 98 % de accuracy. Revisa métricas por clase, pesos de clase, remuestreo solo en entrenamiento y el umbral de decisión.

## 3. Modelos de regresión

### Regresión lineal

**Qué aprende:** busca una combinación ponderada de las variables que aproxime el valor numérico. Cada coeficiente representa el cambio esperado asociado a una variable, manteniendo las demás constantes, bajo los supuestos del modelo.

**Cuándo probarla:** como baseline, cuando las relaciones son aproximadamente lineales o cuando necesitas explicar el efecto de las variables. Es rápida y sus coeficientes ayudan a interpretar.

**Qué debes revisar:** relaciones no lineales, valores atípicos, colinealidad (variables muy relacionadas entre sí) y patrones en los residuos. Un modelo lineal no implica que el mundo sea lineal; describe la aproximación que has elegido.

**Ajustes útiles:** `fit_intercept` incluye un término constante. `Ridge` (`alpha`) reduce el tamaño de coeficientes; `Lasso` (`alpha`) puede llevar algunos a cero; `ElasticNet` combina ambos. Escala las variables antes de regularizar para que la penalización sea comparable.

### Regresión polinomial

**Qué aprende:** añade potencias e interacciones de variables para que una regresión lineal pueda representar curvas (por ejemplo, `x²`). Sigue siendo lineal respecto a los coeficientes, aunque la curva de entrada sea no lineal.

**Cuándo probarla:** con pocas variables numéricas y una curva suave que la recta no captura.

**Ajustes y riesgos:** `degree` controla el grado. Empieza bajo (2 o 3), compara con validación y añade regularización si hace falta. Al subir el grado aparecen muchos términos; el modelo puede oscilar y memorizar ruido, especialmente fuera del rango observado.

### Árbol de regresión

**Qué aprende:** divide los datos mediante reglas del tipo «¿variable < umbral?» y predice un valor medio en cada hoja.

**Cuándo probarlo:** si sospechas umbrales, interacciones o relaciones no lineales y quieres reglas que se puedan visualizar.

**Ajustes clave:** `max_depth` limita la profundidad; `min_samples_leaf` exige un mínimo de ejemplos por hoja; `max_features` limita variables candidatas en cada división. Árboles profundos pueden ajustar demasiado y cambiar mucho ante pequeñas variaciones de los datos.

### Random Forest y Extra Trees

**Qué aprenden:** entrenan muchos árboles diferentes y promedian sus predicciones. El promedio reduce la variabilidad de un árbol individual y suele dar un resultado sólido en datos tabulares.

**Cuándo probarlos:** cuando hay relaciones no lineales, interacciones, escalas variadas y no quieres asumir una fórmula lineal. Son buenos candidatos para comparar con una regresión.

**Ajustes clave:** `n_estimators` (número de árboles), `max_depth`, `min_samples_leaf` y `max_features`. Más árboles suelen estabilizar la predicción, pero no sustituyen controlar árboles demasiado complejos. Extra Trees elige cortes aleatorios (sin optimizar el umbral) y no usa bootstrap: suele ser más rápido y reducir la varianza, a costa de aumentar algo el sesgo.

**Límites:** no necesitan escalado, pero las categóricas suelen requerir codificación. Pueden ser menos interpretables que un árbol único y suelen extrapolar mal más allá del rango observado.

### Gradient Boosting, XGBoost, LightGBM y CatBoost

**Qué aprenden:** construyen árboles de forma secuencial; cada nuevo árbol intenta corregir errores que quedan en los anteriores. Suelen rendir muy bien en conjuntos tabulares medianos o grandes.

**Cuándo probarlos:** cuando importa el rendimiento predictivo en datos tabulares y los patrones no son simples. Compara con Random Forest y un baseline; no des por hecho que el algoritmo más sofisticado gana.

**Ajustes clave:** `learning_rate` regula cuánto aporta cada árbol; `n_estimators` es cuántos árboles se añaden; `max_depth` o `num_leaves` controla complejidad; `subsample` y regularización pueden reducir sobreajuste. Una tasa menor suele requerir más árboles.

**Límites:** hay más hiperparámetros y el ajuste requiere validación. CatBoost facilita el tratamiento de categóricas en su propia implementación; comprueba la API concreta y evita convertirlas a números arbitrarios.

## 4. Modelos de clasificación

### Regresión logística

**Qué aprende:** calcula una puntuación lineal y la transforma en una probabilidad por clase. Aunque se llame regresión, se usa para clasificación.

**Cuándo probarla:** como baseline rápido e interpretable, sobre todo cuando una frontera aproximadamente lineal tiene sentido. También devuelve probabilidades útiles para elegir un umbral.

**Ajustes clave:** `C` controla la regularización inversamente: un `C` menor aplica más regularización y penaliza con más fuerza los coeficientes. `class_weight="balanced"` puede ayudar si las clases están desbalanceadas. Escala numéricas cuando regularizas; codifica categóricas.

**Límites:** la frontera es lineal salvo que añadas transformaciones o interacciones. El umbral 0.5 no es obligatorio: elige según el coste de falsos positivos y falsos negativos.

### K vecinos más cercanos (KNN)

**Qué aprende:** para un caso nuevo, busca los ejemplos más cercanos y vota por sus clases (o promedia sus valores en regresión). No aprende una fórmula compacta; conserva los ejemplos.

**Cuándo probarlo:** con conjuntos pequeños o medianos donde casos cercanos suelen compartir etiqueta y las distancias tienen significado.

**Ajustes clave:** `n_neighbors` (`k`) suaviza la decisión: un k bajo sigue detalles y ruido; uno alto produce fronteras más suaves. `weights="distance"` da más influencia a vecinos cercanos; `metric` define distancia.

**Requisitos y límites:** escala variables y elimina columnas irrelevantes; muchas dimensiones vuelven menos informativas las distancias. Entrenar es sencillo, pero predecir puede ser lento al crecer los datos.

### Árbol de decisión

**Qué aprende:** aplica preguntas sucesivas sobre variables hasta llegar a una clase o probabilidad en una hoja.

**Cuándo probarlo:** si necesitas explicar una decisión mediante reglas o esperas umbrales e interacciones. No necesita escalado.

**Ajustes clave:** `max_depth`, `min_samples_split`, `min_samples_leaf` y `class_weight`. Controlar la profundidad y exigir hojas con varios casos suele mejorar la generalización.

**Límites:** muy profundo puede memorizar; un árbol puede ser inestable. Una importancia de variables basada en impureza no demuestra causalidad.

### Random Forest y Extra Trees

**Qué aprenden:** combinan votos de muchos árboles entrenados con variaciones de los datos y/o de las variables. La combinación suele ser más estable que un árbol solo.

**Cuándo probarlos:** buen candidato general para clasificación tabular con interacciones, no linealidad y escalas variadas.

**Ajustes clave:** `n_estimators`, `max_depth`, `min_samples_leaf`, `max_features` y `class_weight`. Comprueba métricas por clase si existe desbalance.

**Límites:** requieren codificar categóricas en sklearn, son menos transparentes que un árbol pequeño y sus probabilidades pueden requerir calibración si se usarán como probabilidades fiables.

### Boosting de árboles

**Qué aprende:** cada árbol nuevo intenta mejorar la clasificación que dejan los anteriores. XGBoost y LightGBM son implementaciones populares; CatBoost es práctico con categóricas.

**Cuándo probarlo:** cuando tienes datos tabulares y quieres una opción de alto rendimiento, especialmente con relaciones complejas.

**Ajustes clave:** `learning_rate`, número de árboles (`n_estimators`), profundidad/hojas, regularización y `class_weight` o pesos equivalentes. Usa parada temprana con un conjunto de validación cuando esté disponible.

**Límites:** es fácil ajustar demasiado si se prueban configuraciones mirando el test. Evalúa con validación cruzada y compara con alternativas más simples.

### Support Vector Machine (SVM)

**Qué aprende:** busca una frontera que separe clases dejando el mayor margen posible. Con kernel RBF puede representar fronteras curvas.

**Cuándo probarlo:** conjuntos pequeños o medianos, muchas variables y fronteras lineales o moderadamente no lineales. Con margen blando tolera cierto solapamiento entre clases. La versión lineal es útil con muchas columnas, por ejemplo texto vectorizado.

**Ajustes clave:** `C` regula cuánto penaliza errores; en RBF, `gamma` determina el alcance de cada ejemplo. `C` alto y `gamma` alto pueden crear una frontera demasiado ajustada.

**Requisitos y límites:** escala siempre. Con kernel puede ser lento en muchas filas; las probabilidades requieren configuración adicional.

### Naive Bayes

**Qué aprende:** calcula probabilidades de clase suponiendo que las variables son independientes entre sí una vez conocida la clase.

**Cuándo probarlo:** baseline rápido para clasificación de texto con muchas palabras o variables, y cuando hay bastantes columnas frente al número de filas.

**Variantes:** `MultinomialNB` para conteos/frecuencias no negativos; `BernoulliNB` para presencia/ausencia; `GaussianNB` para variables continuas aproximadamente normales.

**Límites:** la independencia rara vez se cumple exactamente. Aun así, puede clasificar bien; verifica sus probabilidades antes de usarlas para decisiones sensibles.

### Perceptrón multicapa (MLP)

**Qué aprende:** combina capas de neuronas y funciones no lineales para aproximar patrones complejos.

**Cuándo probarlo:** hay suficientes datos, relaciones complejas y recursos para ajustar el entrenamiento. En imágenes, audio o texto suelen usarse arquitecturas especializadas de Deep Learning, no un MLP genérico.

**Ajustes clave:** `hidden_layer_sizes` define capas y neuronas; `activation` define la no linealidad; `alpha` regulariza; `learning_rate_init` controla el paso de optimización; `early_stopping` detiene si validación deja de mejorar.

**Requisitos y límites:** escala numéricas, codifica categorías e imputa faltantes. Requiere más decisiones, es menos interpretable y en datasets tabulares pequeños no necesariamente supera a boosting.

## 5. Métodos no supervisados

### K-Means (clustering)

**Qué hace:** asigna cada observación al centro de grupo más cercano y recalcula centros hasta estabilizarse. Busca grupos compactos representados por un centro.

**Cuándo probarlo:** quieres segmentar datos numéricos y esperas grupos más o menos compactos y de tamaño razonable.

**Ajustes y preparación:** `n_clusters` fija el número de grupos; `n_init` ejecuta varias inicializaciones. Escala variables: de lo contrario, una columna con valores grandes puede dominar la distancia. Prueba varios k y revisa tanto métricas como utilidad práctica.

**Límites:** hay que elegir k; outliers y grupos alargados pueden confundirlo. El coeficiente silhouette ayuda a comparar separaciones, pero no decide por sí solo si un segmento es útil.

### DBSCAN (clustering)

**Qué hace:** agrupa regiones densas, y marca como ruido los puntos aislados. No obliga a asignar cada observación a un grupo.

**Cuándo probarlo:** sospechas grupos de forma irregular o quieres identificar ruido/outliers y no sabes cuántos grupos hay.

**Ajustes y preparación:** `eps` define qué tan cerca deben estar dos puntos; `min_samples` cuántos vecinos hacen falta para considerar una zona densa. Escala las variables y prueba parámetros en relación con las distancias observadas.

**Límites:** un mismo `eps` puede no funcionar si los grupos tienen densidades distintas. En alta dimensión las distancias son menos informativas.

### PCA (reducción de dimensión)

**Qué hace:** crea componentes como combinaciones lineales de las variables originales. El primer componente captura la mayor variación posible; el siguiente captura la mayor variación restante, y así sucesivamente.

**Cuándo probarlo:** quieres resumir variables numéricas correlacionadas, visualizar datos en 2D/3D o reducir ruido/columnas antes de otro análisis.

**Ajustes y preparación:** estandariza si las variables usan unidades o escalas distintas. `n_components` fija cuántos componentes conservar o la fracción de varianza objetivo. Examina varianza explicada y cargas para saber qué representa cada componente.

**Límites:** maximizar varianza no garantiza conservar la información más útil para predecir. Los componentes son menos interpretables y PCA no selecciona directamente las variables «más importantes».

## 6. Escoge la métrica según el coste del error

### Para regresión

- **MAE:** error absoluto medio, en unidades del objetivo. Fácil de explicar y menos sensible a errores extremos que RMSE.
- **RMSE:** raíz del error cuadrático medio. Penaliza mucho los errores grandes; úsalo si fallos grandes son especialmente costosos.
- **R²:** compara el modelo con predecir la media. Un valor alto no garantiza ausencia de sesgo; puede ser negativo si el modelo es peor que esa referencia.

### Para clasificación

- **Accuracy:** porcentaje total de aciertos. Tiene sentido si las clases están equilibradas y los errores tienen costes parecidos.
- **Precision:** entre los casos predichos positivos, cuántos son realmente positivos. Priorízala cuando falsas alarmas cuestan caro.
- **Recall:** entre los positivos reales, cuántos detectas. Priorízalo cuando dejar pasar positivos cuesta caro.
- **F1:** combina precision y recall; útil como resumen, pero revisa ambas métricas por separado.
- **ROC-AUC:** es independiente del umbral y mide la probabilidad de que un positivo aleatorio quede ordenado por encima de un negativo aleatorio. Con positivos muy raros, mira también PR-AUC.
- **Matriz de confusión:** muestra aciertos y tipos de error; traduce las celdas a consecuencias reales del proyecto.

Una probabilidad no es automáticamente una decisión: decide el umbral con el coste de cada error y, si hace falta, calibra las probabilidades.

## 7. Árbol de decisión rápido

1. **¿Tienes objetivo conocido?** Si no, explora PCA para resumir dimensiones o K-Means/DBSCAN para buscar grupos.
2. **¿Predices un número o una clase?** Elige regresión o clasificación y define la métrica antes de comparar.
3. **¿Necesitas explicar con claridad?** Empieza por un modelo lineal o un árbol pequeño. Mantén una versión sencilla como referencia.
4. **¿Tienes datos tabulares y patrones complejos?** Prueba Random Forest y boosting. Compara en la misma validación.
5. **¿Tienes pocas filas o muchas columnas?** Prueba modelos regularizados, SVM lineal o Naive Bayes para texto. Simplifica variables y valida con cuidado.
6. **¿El modelo usa distancias o magnitudes?** Escala antes. KNN, SVM y K-Means se basan en distancias; PCA y los modelos regularizados también requieren escalado, pero porque la varianza o la penalización dependen de las unidades.
7. **¿Hay tiempo o grupos repetidos?** Separa y valida respetando esa estructura.
8. **¿El modelo mejora el baseline en validación y resuelve la necesidad del proyecto?** Si no, revisa datos, objetivo, variables y métrica antes de aumentar complejidad.

## 8. Señales para diagnosticar

- **Train mucho mejor que validación:** probablemente sobreajuste. Reduce profundidad/complejidad, regulariza, consigue más datos o revisa fugas.
- **Train y validación malos:** probablemente subajuste, variables poco informativas o preparación incorrecta. Prueba una representación o modelo más flexible.
- **Métricas cambian mucho entre pliegues:** puede haber pocos datos, grupos distintos o una partición inadecuada. Revisa la forma de validar.
- **Accuracy alta pero positivos no detectados:** desbalance. Mira recall, precision, matriz de confusión y umbral.
- **Muy buen resultado que parece imposible:** investiga fuga de información, duplicados y variables posteriores al evento.

**Idea final:** el mejor modelo para el proyecto es el que generaliza a los datos futuros, mejora una referencia sencilla y permite tomar una decisión útil con un nivel de explicación y coste aceptables.
